# Step 2: Fine-tune the verdict model
**Input to the model:** one `claim` + one `evidence` passage, as a text pair.
**Output:** 3 scores → `SUPPORTS`, `REFUTES`, `NEI`.

**Plan**
1. Load `unified.parquet` from step 1.
2. Train on PUBHEALTH-train + SciFact-train. SciFact is small, so it is repeated ×3 so PUBHEALTH doesn't drown it out.
3. Weight the loss so the rare `NEI` class still matters.
4. Pick the best epoch by macro-F1 on PUBHEALTH-val + SciFact-val.
5. Evaluate once on PUBHEALTH-test and **HealthVer (never trained on)**. Do not tune on HealthVer.
6. Save the model and the raw scores (step 5 needs them for calibration).

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
%pip install -q "transformers>=4.46,<5" accelerate sentencepiece protobuf scikit-learn

In [ ]:
import os, shutil, random, numpy as np, pandas as pd, torch, torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

# ---- config ----
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 384          # abstracts are ~300 tokens; shorter = faster
EPOCHS = 3
LR = 2e-5
BATCH = 16
SCIFACT_REPEAT = 3     # oversample the small, in-domain dataset
SEED = 42
SMOKE = False          # True = tiny quick run just to check the code works
# ----------------
LABELS = ["SUPPORTS", "REFUTES", "NEI"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not SMOKE:
    assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (smoke test)")

## Google Drive (so a disconnect doesn't cost you the model)
Checkpoints, the final model and the scores are written straight to Drive. **Approve the Drive pop-up** when it appears.
If the runtime dies mid-training, reconnect, rerun all cells, and training resumes from the last saved epoch.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:          # not on Colab (local test only)
    DRIVE = "veritas_local"
os.makedirs(DRIVE, exist_ok=True)
CKPT_DIR = f"{DRIVE}/ckpt"
print("saving to:", DRIVE)

## Load data
Looks for `unified.parquet` here, then in Drive, otherwise asks you to upload it from step 1.

In [ ]:
if not os.path.exists("unified.parquet"):
    if os.path.exists(f"{DRIVE}/unified.parquet"):
        shutil.copy(f"{DRIVE}/unified.parquet", "unified.parquet")
    else:
        from google.colab import files
        files.upload()      # choose unified.parquet from step 1
if not os.path.exists(f"{DRIVE}/unified.parquet"):
    shutil.copy("unified.parquet", f"{DRIVE}/unified.parquet")   # keep a copy on Drive
df = pd.read_parquet("unified.parquet")

train_df = pd.concat([
    df[(df.source == "pubhealth") & (df.split == "train")],
    pd.concat([df[(df.source == "scifact") & (df.split == "train")]] * SCIFACT_REPEAT),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

ph_val  = df[(df.source == "pubhealth") & (df.split == "validation")].reset_index(drop=True)
sf_val  = df[(df.source == "scifact")   & (df.split == "validation")].reset_index(drop=True)
ph_test = df[(df.source == "pubhealth") & (df.split == "test")].reset_index(drop=True)
hv      = df[df.source == "healthver"].reset_index(drop=True)

if SMOKE:
    train_df = train_df.sample(120, random_state=0).reset_index(drop=True)
    ph_val, sf_val, ph_test, hv = [x.head(40) for x in (ph_val, sf_val, ph_test, hv)]

val_df = pd.concat([ph_val, sf_val]).reset_index(drop=True)
print({k: len(v) for k, v in dict(train=train_df, val=val_df, ph_test=ph_test, healthver=hv).items()})
print(train_df.label_name.value_counts())

## Class weights
`NEI` is rare, so a plain loss lets the model ignore it. Weight = (N / (3 × class count)) ^ 0.5.
The square root keeps the boost gentle. Full inverse-frequency weights tend to make the model over-predict the rare class.

In [ ]:
counts = train_df.label.value_counts().reindex([0, 1, 2]).values
cw = torch.tensor((len(train_df) / (3 * counts)) ** 0.5, dtype=torch.float)
print(dict(zip(LABELS, cw.numpy().round(2))))

## Tokenize
Each example becomes `[CLS] claim [SEP] evidence [SEP]`. `truncation="only_second"` cuts the evidence, never the claim.
Padding is done per batch (dynamic), so short PUBHEALTH rows are cheap.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(d):
    ds = Dataset.from_pandas(d[["claim", "evidence", "label"]].rename(columns={"label": "labels"}),
                             preserve_index=False)
    return ds.map(lambda b: tok(b["claim"], b["evidence"], truncation="only_second", max_length=MAX_LEN),
                  batched=True, remove_columns=["claim", "evidence"])

train_ds, val_ds = make_ds(train_df), make_ds(val_df)
print(train_ds)

## Model + weighted trainer
Only the loss is customised: cross-entropy with the class weights above.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = F.cross_entropy(out.logits.float(), labels,
                               weight=self.class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "acc": float((preds == p.label_ids).mean())}

args = TrainingArguments(
    output_dir=CKPT_DIR, num_train_epochs=1 if SMOKE else EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
    warmup_ratio=0.1, weight_decay=0.01, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    group_by_length=True, logging_steps=50, report_to="none", seed=SEED)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                          class_weights=cw)

## Train (about 30–60 min on a T4)
Resumes automatically if a checkpoint already exists on Drive.

In [ ]:
resume = os.path.isdir(CKPT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(CKPT_DIR))
print("resuming from checkpoint" if resume else "starting fresh")
trainer.train(resume_from_checkpoint=True if resume else None)

## Evaluate
Shown per dataset so you can see where it is weak. **HealthVer is the honest number**: the model never saw it.

In [ ]:
def report(name, d):
    out = trainer.predict(make_ds(d))
    y, pred = out.label_ids, out.predictions.argmax(-1)
    print(f"\n===== {name}  (n={len(d)})  macro-F1 = {f1_score(y, pred, average='macro'):.3f}  acc = {(y == pred).mean():.3f}")
    print(classification_report(y, pred, labels=[0, 1, 2], target_names=LABELS, zero_division=0))
    print("confusion (rows = true, cols = pred):\n", confusion_matrix(y, pred, labels=[0, 1, 2]))
    return out.predictions, y

res = {
    "pubhealth_val":  report("PUBHEALTH val",  ph_val),
    "scifact_val":    report("SciFact val",    sf_val),
    "pubhealth_test": report("PUBHEALTH test", ph_test),
    "healthver":      report("HealthVer (held out)", hv),
}

## Save (straight to Drive)
The scores go to `logits.npz` for step 5 (calibration + abstention). The model goes to `verdict_model/`.

In [ ]:
np.savez(f"{DRIVE}/logits.npz", **{f"{k}_logits": v[0] for k, v in res.items()},
                                **{f"{k}_labels": v[1] for k, v in res.items()})
trainer.save_model(f"{DRIVE}/verdict_model"); tok.save_pretrained(f"{DRIVE}/verdict_model")
print(sorted(os.listdir(DRIVE)))

## What to check before Step 3
1. **Macro-F1 on SciFact val vs PUBHEALTH val.** A big gap means one dataset is dominating.
2. **HealthVer macro-F1 vs the in-domain numbers.** That drop is a headline number for your pitch.
3. **NEI recall.** If it is near 0, raise the class weight (drop the square root).
4. Loss going down but val macro-F1 flat after epoch 1 means overfitting. Use fewer epochs.